# Training CNN Model with PyTorch for Land Use Classification
Huấn luyện mô hình CNN với PyTorch cho phân loại sử dụng đất từ dữ liệu Sentinel-1 và Sentinel-2

In [ ]:
%%time
%matplotlib inline

import importlib
import new_import_ODC  

importlib.reload(new_import_ODC)

from new_import_ODC import *

In [ ]:
# Kiểm tra GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    device = 'cuda'
else:
    print("Using CPU for training")
    device = 'cpu'

print(f"\nDevice sẽ dùng: {device}")

In [ ]:
%%time
# Cấu hình Daskgateway
cluster, client = notebook_utils.initialize_dask(use_gateway=True, workers=(1, 10))
# Khai báo 1 Datacube là dc
dc = datacube.Datacube()

# Cấu hình truy cập dịch vụ S3
configure_s3_access(aws_unsigned=False, requester_pays=True, client=client)

client

In [ ]:
## cấu hình thời gian lấy ảnh và tọa độ
date_range = ("2022-09-01", "2023-10-01")
longtitude_range = (105.5, 106.4)
latitude_range = (9.2, 10.0)

coordinates = (longtitude_range, latitude_range)

In [ ]:
## truy vấn ảnh vệ tinh sen2
data = load_data(dc, date_range, longtitude_range, latitude_range)
notebook_utils.heading(notebook_utils.xarray_object_size(data))
display(data)

In [ ]:
%%time
# Tiến hành loại bỏ các vị trí bị mây ảnh hưởng
result = mask_clean(data)
progress(result)

In [ ]:
# Tiến hành tính toán NDVI
ds1 = calculate_indices(result, index="NDVI", satellite_mission="s2")
ndvi = ds1["NDVI"]
display(ndvi)

In [ ]:
## Hiển thị ảnh NDVI chưa điền các giá trị mây (chưa fill nan)
plt.imshow(ndvi.isel(time=6))

In [ ]:
# Thiết lập giá trị trung bình mùa vụ để xử lý các điểm ảnh bị mây dựa vào sự thay đổi theo mùa
time_split = [
    slice("2022-09-01", "2023-01-01"),
    slice("2023-01-01", "2023-05-01"),
    slice("2023-05-01", "2023-07-01"),
    slice("2023-07-01", "2023-10-01"),
]

# Điền mây ở các vị trí mang giá trị nan (fill nan)
fill_nan_ndvi = fill_nan(ndvi, time_split)

# In kết quả ảnh NDVI đã điền mây (đã fill nan)
plt.imshow(fill_nan_ndvi.isel(time=6))

In [ ]:
%%time
## tính ndvi theo tháng
average_ndvi = fill_nan_ndvi.resample(time="1M").mean().persist()
progress(average_ndvi)

# compute average_ndvi
average_ndvi = average_ndvi.compute()

In [ ]:
#Load dữ liệu ảnh Sentinel 1
dsvh, dsvv = load_data_sen1(dc, date_range, coordinates)
average_vv = calculate_average(dsvv, time_pattern='1M')
average_vh = calculate_average(dsvh, time_pattern='1M')

In [ ]:
## cấu hình bộ dữ liệu điểm huấn luyện mô hình (train file)
train_path = "train/ST_training data_updated_1130points_new.shp"  # đường dẫn shp file train

## load dữ liệu điểm huấn luyện mô hình (train file)
train = load_train_data(train_path)
train.head()

# cấu hình nhãn dữ liệu 
label_mapping = {
    "Lua tom": "0",
    "Lua": "1",
    "CHN": "2",
    "CLN": "3",
    "TS": "4",
    "Song": "5",
    "Dat xay dung": "6",
    "Rung": "7",
}

# xây dựng tập dữ liệu (dataset) chứa dữ liệu VH, VV, NDVI
datasets = get_data_sen1_and_sen2(train, average_ndvi, average_vh, average_vv)

# chia tập dữ liệu thành các phần theo tỉ lệ 80(80-20)-20 tương ứng với tập train, validate, test
X_train, X_val, X_test, y_train, y_val, y_test = split_train_data(
    train, label_mapping, datasets
)

## Huấn luyện mô hình CNN với PyTorch

In [ ]:
%%time
# Huấn luyện mô hình CNN với PyTorch
print("🔷 Bắt đầu huấn luyện CNN model với PyTorch...\n")

cnn_model, history, scaler = train_cnn_pytorch(
    X_train, X_val, X_test, 
    y_train, y_val, y_test,
    num_classes=8,
    epochs=100,
    batch_size=32,
    learning_rate=1e-3,
    device=device,
    verbose=True
)

In [ ]:
# Vẽ đồ thị huấn luyện
plot_pytorch_training_history(history)

In [ ]:
# Lưu mô hình CNN PyTorch
save_pytorch_model(cnn_model, scaler, model_name="model_cnn_pytorch.pth")

In [ ]:
# Hiển thị model architecture
print("\n📐 Model Architecture:")
print(cnn_model)

# Đếm số parameters
total_params = sum(p.numel() for p in cnn_model.parameters())
trainable_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f"\n📊 Model Parameters:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

In [ ]:
# đóng client, cluster
client.close()
cluster.close()